# Experimentul 9: ZZ feature map rar (sparse) și reducerea CNOT

**Pornim de la Exp. 3** (ZZ ×1, ansatz ×3, 4 qubiți, 32 de parametri, readout ⟨Z₀⟩). Singura modificare: **ce perechi de qubiți primesc o interacțiune ZZ** în feature map. Fiecare interacțiune ZZ este blocul `CNOT(i, j) → RZ(2(π − xᵢ)(π − xⱼ)) pe j → CNOT(i, j)`; eliminarea unei perechi elimină blocul întreg (2 CNOT + 1 RZ).

| Topologie | Perechi ZZ (în ordinea din feature map-ul oficial) | Interacțiuni |
|---|---|---:|
| **FULL** (Exp. 3, referință) | (0,1), (0,2), (0,3), (1,2), (1,3), (2,3) | 6 |
| **STAR** (centru = qubitul măsurat 0) | (0,1), (0,2), (0,3) | 3 |
| **LINEAR** | (0,1), (1,2), (2,3) | 3 |

## Ipoteze (de testat, nu presupuse)
- **H1:** cu 6 CNOT mai puțin (și 15 locuri de zgomot mai puțin, estimat), scorurile sunt mai puțin perturbate de zgomot, iar AUPRC la p = 0.01 / 0.05 scade mai puțin.
- **H0 / alternativă:** produsele (π − xᵢ)(π − xⱼ) eliminate purtau informație utilă; STAR / LINEAR pierd deja la p = 0 mai mult decât câștigă în robustețe.

## Protocol (identic cu Exp. 3)
Split `split_indices.npz`, `undersample(X_train, y_train, seed)` 1:1 (590 de exemple), `AngleEncoder(4, seed)` fitat pe setul 1:1 (**același** encoder și **aceleași** unghiuri pentru toate topologiile), `init_weights(seed)` și ordinea mini-batch-urilor `torch.Generator().manual_seed(seed)` identice cu Exp. 3 (ansatz-ul nu se schimbă), Adam lr 0.05, BCE, batch 32, 30 de epoci, antrenare fără zgomot, epoca aleasă după AUPRC pe validation fără zgomot. Evaluare cu modelul oficial de zgomot (`noisy_qnode`: decompoziție în `NATIVE_GATES`, `DepolarizingChannel(p)` după fiecare poartă, pe fiecare qubit, `default.mixed`) la p = 0, 0.001, 0.01, 0.05, prag max-F1 pe validation la același p.

**FULL nu se reantrenează**: folosim ponderile Exp. 3. Scorurile lui sub zgomot se recalculează aici în float64 (aceleași condiții numerice ca STAR / LINEAR, vezi Exp. 8 §6), fără a suprascrie rezultatele istorice.

**Doar TRAIN și VALIDATION.** Testul nu se folosește: din el păstrăm doar indicii, pentru verificarea splitului. Fără rotation fusion (Exp. 8), fără noise-aware training, ansatz și număr de qubiți neschimbate.

## 1. Configurație și imports

**Seeds.** `set_seed` fixează numpy și torch; amestecarea folosește un `torch.Generator` dedicat (ca în Exp. 3). Simulatoarele PennyLane rulează cu `shots=None` (valori exacte); singura excepție este numărarea execuțiilor parameter-shift cu `qml.Tracker`, pe un dispozitiv cu 1024 shots și `seed` fix. Dtype implicit torch: float64 (în float32, `default.mixed` are erori ~1e-7 chiar la p = 0; Exp. 8 §6).

In [ ]:
import hashlib
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- singura modificare față de Exp. 3: perechile ZZ ----
OFFICIAL_PAIRS = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]   # ordinea din feature_map oficial: i < j, lexicografic
TOPOLOGIES = {
    "full": [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)],       # Exp. 3, referință (ponderi refolosite)
    "star": [(0, 1), (0, 2), (0, 3)],                               # qubitul 0 = centru și qubit măsurat
    "linear": [(0, 1), (1, 2), (2, 3)],
}
NEW_TOPOLOGIES = ["star", "linear"]   # antrenate de la zero în Exp. 9

# ---- fix, identic cu Exp. 3 ----
SPLIT_SEED = 42
N_QUBITS = 4
FEATURE_MAP_REPS = 1
ANSATZ_REPS = 3               # (3 + 1) * 4 * 2 = 32 parametri
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.05
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
NOISE_LEVELS = [0.001, 0.01, 0.05]
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, *NOISE_LEVELS]

# ---- verificări ----
TORCH_DEFAULT_DTYPE = torch.float64
EXACT_TOL = 1e-12             # reproducerea Exp. 3 / Exp. 8 în aceleași condiții numerice (float64)
REPRO_TOL = 1e-6              # reproducerea scorurilor istorice Exp. 3 calculate cu float32 implicit (Exp. 8 §6)
SMOKE_ROWS = 2000             # testul de fum: primele tranzacții din validation, seed 0, p = 0.05
SMOKE_EPOCHS = 2

EXPERIMENT_NAME = "sparse_zz_feature_map"
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp9"
WEIGHTS_DIR = RESULTS_DIR / "weights"
PLOTS_DIR = RESULTS_DIR / "plots"
EXP3_DIR = REPO_ROOT / "results" / "result_exp3"   # doar citit: ponderi, istoric, praguri, metrici
EXP8_DIR = REPO_ROOT / "results" / "result_exp8"   # doar citit: Exp. 3 recalculat în float64 (Scenariul A)
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

TOPO_LABELS = {"full": "FULL ZZ (Exp. 3)", "star": "STAR ZZ", "linear": "LINEAR ZZ"}
TOPO_COLORS = {"full": "#2a78d6", "star": "#eb6834", "linear": "#1baf7a"}   # primele 3 sloturi ale paletei validate
TOPO_MARKERS = {"full": "o", "star": "s", "linear": "D"}
SEED_DOT_COLOR, CNOT_COLOR = "#8a8f98", "#c9ccd1"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


for name, pairs in TOPOLOGIES.items():   # perechile rămase păstrează ordinea relativă din feature map-ul oficial
    assert [p for p in OFFICIAL_PAIRS if p in pairs] == pairs, f"{name}: ordinea perechilor diferă de cea oficială"
assert TOPOLOGIES["full"] == OFFICIAL_PAIRS == [(i, j) for i in range(N_QUBITS) for j in range(i + 1, N_QUBITS)]

set_seed(SPLIT_SEED)
torch.set_default_dtype(TORCH_DEFAULT_DTYPE)
print(f"topologii: { {k: len(v) for k, v in TOPOLOGIES.items()} }   seeds: {SEEDS}   p: {P_ALL}   dtype implicit torch: {torch.get_default_dtype()}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

Identic cu Exp. 3: `split_indices.npz`, verificat față de reconstrucția oficială. Din test păstrăm doar indicii pentru verificarea splitului; nici features, nici etichete.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for path in [DATA_PATH, SPLIT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
assert (len(train_idx), int(y_train.sum())) == (170883, 295)
assert (len(val_idx), int(y_val.sum())) == (56962, 99)
del X_all, y_all, df, _trval, _test, _train, _val, test_idx   # testul nu se folosește în Exp. 9

print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())} fraude)   test: neatins")

## 3. Funcțiile oficiale și cele trei circuite

`feature_map`, `ansatz`, `vqc`, `undersample`, `AngleEncoder`, `init_weights`, `choose_threshold`, `evaluate`, `noisy_qnode`, `noisy_predict`, `noise_locations` sunt copiate neschimbate din Exp. 3 / notebookul oficial. Singura funcție nouă, `make_feature_map(pairs)`, este `feature_map` oficial în care bucla dublă `for i: for j > i` este înlocuită cu `for (i, j) in pairs`; porțile de codificare `H, RZ(2xᵢ)`, formula unghiului ZZ și ansatz-ul sunt aceleași. Funcțiile de antrenare și predicție primesc modelul ca argument.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


def make_feature_map(pairs):
    """The official ZZ feature map with ZZ interactions only on `pairs` (kept in the given order)."""
    def feature_map_pairs(x):
        for _ in range(FEATURE_MAP_REPS):
            for i in range(N_QUBITS):
                qml.Hadamard(wires=i)
                qml.RZ(2 * x[..., i], wires=i)
            for i, j in pairs:
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])
    return feature_map_pairs


def make_vqc(pairs):
    fm = make_feature_map(pairs)

    def vqc_pairs(x, weights):
        fm(x)
        ansatz(weights)
        return qml.expval(qml.PauliZ(0))
    return vqc_pairs


FEATURE_MAPS = {t: make_feature_map(p) for t, p in TOPOLOGIES.items()}
CIRCUITS = {t: make_vqc(p) for t, p in TOPOLOGIES.items()}
dev4 = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
QNODES = {t: qml.QNode(fn, dev4, interface="torch", diff_method="backprop") for t, fn in CIRCUITS.items()}
MODELS = {t: (lambda q: (lambda x, weights: (1 - q(x, weights)) / 2))(q) for t, q in QNODES.items()}   # fraud_probability per topologie


def readout(expval):
    return (1 - expval) / 2


def init_weights(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.nn.Parameter(2 * np.pi * torch.rand(ANSATZ_REPS + 1, N_QUBITS, 2, generator=generator, dtype=torch.float64))


def predict(Z, weights, model, chunk=20000):
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def train_vqc_run(Z_tr, y_tr, Z_valid, y_valid, initial_weights, seed, model, epochs=EPOCHS):
    """The training loop of Exp. 1-3, with the model (fraud_probability of a topology) as an argument."""
    set_seed(seed)
    shuffle_generator = torch.Generator().manual_seed(seed)
    weights = torch.nn.Parameter(initial_weights.detach().clone())   # copie independentă
    optimizer = torch.optim.Adam([weights], lr=LEARNING_RATE)
    loss_fn = torch.nn.BCELoss()
    X_t = torch.tensor(Z_tr)
    y_t = torch.tensor(y_tr, dtype=torch.float64)
    history, best_ap, best_weights, best_epoch = [], -1.0, None, None
    orders = hashlib.sha256()
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_t), generator=shuffle_generator)
        orders.update(order.numpy().tobytes())
        losses = []
        for k in range(0, len(X_t), BATCH_SIZE):
            idx = order[k:k + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(model(X_t[idx], weights), y_t[idx])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        val_ap = average_precision_score(y_valid, predict(Z_valid, weights, model))
        history.append({"epoch": epoch, "train loss": float(np.mean(losses)), "validation AUPRC": float(val_ap)})
        if val_ap > best_ap:
            best_ap, best_weights, best_epoch = val_ap, weights.detach().clone(), epoch
    return {"weights": best_weights, "history": history, "best_epoch": best_epoch, "best_val_ap": float(best_ap),
            "time_train_s": time.perf_counter() - start, "batch_order_sha256": orders.hexdigest()}


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS):
    """QNode of circuit_fn under the challenge's noise model: native gates, then a depolarizing channel of
    probability p after every gate, on each of its qubits. Differentiable with backprop."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=NATIVE_GATES)


def noisy_predict(Z, weights, p, circuit_fn=vqc, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def noise_locations(circuit_fn, *args, n_wires=N_QUBITS):
    """Number of depolarizing channels the noise model inserts: one per qubit of every native gate."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires)), gate_set=NATIVE_GATES)
    return sum(size * count for size, count in qml.specs(qnode)(*args)["resources"].gate_sizes.items())

### Verificarea circuitelor

1. **FULL = Exp. 3, poartă cu poartă:** tape-ul decompus al `CIRCUITS["full"]` este identic (`qml.equal`, aceleași unghiuri) cu cel al `vqc` oficial, pe 20 de intrări.
2. **STAR / LINEAR = FULL fără blocurile excluse:** tape-ul lor este exact tape-ul FULL din care s-au scos cele 3 porți (`CNOT, RZ, CNOT`) ale fiecărei perechi excluse; tot restul (codificare, blocurile rămase în aceeași ordine, ansatz) este identic.
3. **Check 6.1 per topologie:** starea feature map-ului = aceeași stare construită cu `qml.IsingZZ` pe aceleași perechi (fidelitate 1).
4. **Ansatz (check 6.2) și canalul depolarizant (check 10.1)** ca în Exp. 3.

In [ ]:
def tape_ops(circuit_fn, x, weights):
    """Operations of circuit_fn after decomposition into NATIVE_GATES (numpy inputs: one transaction)."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, dev4), gate_set=NATIVE_GATES)
    return qml.workflow.construct_tape(qnode)(x, weights).operations


def same_ops(a, b):
    return len(a) == len(b) and all(qml.equal(o1, o2) for o1, o2 in zip(a, b))


def state_of(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.QNode(full, dev4)()


def isingzz_feature_map(pairs):
    def fm(x):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[i], wires=i)
        for i, j in pairs:
            qml.IsingZZ(2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])
    return fm


rng_check = np.random.default_rng(SPLIT_SEED)
X_check = rng_check.uniform(0, np.pi, (20, N_QUBITS))
W_check = rng_check.uniform(0, 2 * np.pi, (20, ANSATZ_REPS + 1, N_QUBITS, 2))
N_ENCODING_OPS, OPS_PER_ZZ = 2 * N_QUBITS, 3
checks = []
for x, w in zip(X_check, W_check):
    official = tape_ops(vqc, x, w)
    assert same_ops(official, tape_ops(CIRCUITS["full"], x, w)), "FULL nu reproduce poartă cu poartă vqc-ul oficial"
    for t in NEW_TOPOLOGIES:
        blocks = {pair: official[N_ENCODING_OPS + OPS_PER_ZZ * k: N_ENCODING_OPS + OPS_PER_ZZ * (k + 1)] for k, pair in enumerate(OFFICIAL_PAIRS)}
        assert all([op.name for op in b] == ["CNOT", "RZ", "CNOT"] and tuple(b[0].wires) == pair for pair, b in blocks.items())
        expected = official[:N_ENCODING_OPS] + sum((blocks[pair] for pair in TOPOLOGIES[t]), []) + official[N_ENCODING_OPS + OPS_PER_ZZ * len(OFFICIAL_PAIRS):]
        assert same_ops(expected, tape_ops(CIRCUITS[t], x, w)), f"{t}: circuitul nu este FULL fără blocurile excluse"
checks.append({"verificare": "FULL = vqc oficial (Exp. 3), poartă cu poartă", "intrări": len(X_check), "trecut": True})
checks.append({"verificare": "STAR / LINEAR = FULL fără blocurile ZZ excluse (ordinea și restul identice)", "intrări": len(X_check), "trecut": True})

for t, pairs in TOPOLOGIES.items():   # 6.1
    fidelity = [abs(np.vdot(state_of(isingzz_feature_map(pairs), x), state_of(FEATURE_MAPS[t], x))) ** 2 for x in X_check]
    assert np.allclose(fidelity, 1.0, atol=1e-12)
    checks.append({"verificare": f"6.1 {t}: feature map = IsingZZ pe aceleași perechi (fidelitate)", "intrări": len(X_check),
                   "trecut": True, "eroare max": float(np.max(np.abs(np.array(fidelity) - 1)))})

w_lin = torch.linspace(0, 3, (ANSATZ_REPS + 1) * N_QUBITS * 2, dtype=torch.float64).reshape(ANSATZ_REPS + 1, N_QUBITS, 2)   # 6.2
probabilities = np.abs(qml.math.to_numpy(state_of(ansatz, w_lin))) ** 2
assert np.allclose(probabilities[[0, 5, 10, 15]], [0.049016, 0.01134, 0.037841, 0.048166], atol=1e-5)


@qml.qnode(qml.device("default.mixed", wires=1))   # 10.1
def noisy_ry(theta, p):
    qml.RY(theta, wires=0)
    qml.DepolarizingChannel(p, wires=0)
    return qml.expval(qml.PauliZ(0))


for theta, p in [(0.3, 0.0), (1.0, 0.1), (2.5, 0.3), (0.7, 0.75)]:
    assert np.isclose(noisy_ry(theta, p), (1 - 4 * p / 3) * np.cos(theta))
checks += [{"verificare": "6.2 ansatz", "intrări": 1, "trecut": True}, {"verificare": "10.1 canal depolarizant", "intrări": 4, "trecut": True}]
print("✅ FULL = Exp. 3 poartă cu poartă; STAR / LINEAR = FULL fără blocurile excluse; checks 6.1 (toate topologiile), 6.2, 10.1")

## 4. Structura circuitelor și execuțiile parameter-shift

Numărăm pe tape-ul decompus în `NATIVE_GATES` (nimic nu este hardcodat): porți totale, porți de 1 / 2 qubiți, CNOT, interacțiuni ZZ, adâncime (calculată din tape și verificată cu `qml.specs`), locuri de zgomot (verificate cu `noise_locations` oficial), separat pentru feature map și ansatz. Execuțiile parameter-shift per exemplu sunt **măsurate** cu `qml.Tracker` exact ca în Exp. 3 (mini-batch de 32 din setul 1:1 al seed-ului 42, `init_weights(0)`, dispozitiv cu 1024 shots). Numărăm și parametrii cu gradient nenul (backprop, aceleași ponderi, 590 de exemple).

In [ ]:
def circuit_depth(ops):
    depth = {}
    for op in ops:
        level = 1 + max(depth.get(w, 0) for w in op.wires)
        for w in op.wires:
            depth[w] = level
    return max(depth.values())


def gate_stats(ops, prefix=""):
    one_q = sum(len(op.wires) == 1 for op in ops)
    two_q = sum(len(op.wires) == 2 for op in ops)
    return {f"{prefix}porți": len(ops), f"{prefix}porți 1 qubit": one_q, f"{prefix}porți 2 qubiți": two_q,
            f"{prefix}CNOT": sum(op.name == "CNOT" for op in ops), f"{prefix}locuri de zgomot": one_q + 2 * two_q}


X_bal_check, y_bal_check = undersample(X_train, y_train, SPLIT_SEED)
Z_check = AngleEncoder(N_QUBITS, SPLIT_SEED).fit(X_bal_check).transform(X_bal_check)
batch_x, batch_y = torch.tensor(Z_check[:BATCH_SIZE]), torch.tensor(y_bal_check[:BATCH_SIZE], dtype=torch.float64)
w_probe = init_weights(SEEDS[0])
x_probe = Z_check[0]

structure_rows, circuit_ops = [], {}
for t, fn in CIRCUITS.items():
    ops = tape_ops(fn, x_probe, w_probe.detach().numpy())
    fm_ops = qml.workflow.construct_tape(qml.QNode(lambda x, f=FEATURE_MAPS[t]: (f(x), qml.state())[1], dev4))(x_probe).operations
    assert all(op.name in NATIVE_GATES for op in ops) and all(op.name in NATIVE_GATES for op in fm_ops)
    assert same_ops(fm_ops, ops[:len(fm_ops)])
    an_ops = ops[len(fm_ops):]
    resources = qml.specs(qml.transforms.decompose(qml.QNode(fn, dev4), gate_set=NATIVE_GATES))(torch.tensor(x_probe), w_probe)["resources"]
    row = {"topologie": t, "perechi ZZ": " ".join(f"({i},{j})" for i, j in TOPOLOGIES[t]), "interacțiuni ZZ": len(TOPOLOGIES[t]),
           **gate_stats(ops), "adâncime": circuit_depth(ops), **gate_stats(fm_ops, "feature map: "),
           "feature map: adâncime": circuit_depth(fm_ops), **gate_stats(an_ops, "ansatz: "),
           **{f"#{g}": sum(op.name == g for op in ops) for g in ["Hadamard", "RY", "RZ", "CNOT"]}}
    assert row["adâncime"] == resources.depth and row["porți"] == sum(resources.gate_types.values())
    assert row["locuri de zgomot"] == noise_locations(fn, torch.tensor(x_probe), w_probe)
    assert row["feature map: CNOT"] == 2 * len(TOPOLOGIES[t]), "fiecare interacțiune ZZ trebuie să aibă exact 2 CNOT"

    # execuții parameter shift măsurate (ca în Exp. 3) și parametri cu gradient nenul
    shot_device = qml.device("default.qubit", wires=N_QUBITS, seed=SPLIT_SEED)
    hardware_like = qml.set_shots(qml.QNode(fn, shot_device, interface="torch", diff_method="parameter-shift"), shots=SHOTS_PER_EXECUTION)
    w_tracker = init_weights(SEEDS[0])
    with qml.Tracker(shot_device) as tracker:
        loss = torch.nn.BCELoss()(readout(hardware_like(batch_x, w_tracker)), batch_y)
        loss.backward()
    w_grad = init_weights(SEEDS[0])
    torch.nn.BCELoss()(MODELS[t](torch.tensor(Z_check), w_grad), torch.tensor(y_bal_check, dtype=torch.float64)).backward()
    per_example = tracker.totals["executions"] / len(batch_x)
    row.update({"parametri antrenabili": w_probe.numel(), "execuții / exemplu (nominal 1 + 2P)": 1 + 2 * w_probe.numel(),
                "execuții / exemplu (Tracker, parameter shift)": per_example,
                "parametri deplasați de parameter shift (Tracker)": (per_example - 1) / 2,
                "parametri cu gradient nenul": int((w_grad.grad.abs() > 1e-12).sum())})
    structure_rows.append(row)
    circuit_ops[t] = ops

circuit_comparison = pd.DataFrame(structure_rows).set_index("topologie")
exp3_ref = pd.read_csv(EXP3_DIR / "circuit_structure.csv", index_col=0).loc["Exp. 3 (ZZ ×1)"]
full_row = circuit_comparison.loc["full"]
assert (full_row["porți"], full_row["CNOT"], full_row["adâncime"], full_row["locuri de zgomot"], full_row["execuții / exemplu (Tracker, parameter shift)"]) == (
    exp3_ref["porți native"], exp3_ref["CNOT"], exp3_ref["adâncime"], exp3_ref["locuri de zgomot"], exp3_ref["execuții / exemplu (Tracker, parameter shift)"]), "FULL ≠ structura salvată a Exp. 3"
STRUCT = circuit_comparison
N_PARAMS = int(full_row["parametri antrenabili"])
EXECUTIONS_PER_SAMPLE = 1 + 2 * N_PARAMS
print("✅ FULL reconstruit = structura salvată a Exp. 3")
display(circuit_comparison.T)

## 5. Configurația și directorul de rezultate

`config.json` păstrează perechile ZZ ale fiecărei topologii și ordinea exactă a operațiilor (tape-ul decompus). O rulare ulterioară cu altă configurație se oprește în loc să suprascrie.

In [ ]:
def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


def op_sequence(ops):
    return [f"{op.name}{list(op.wires)}" for op in ops]


config = {
    "experiment": EXPERIMENT_NAME,
    "base_model": "Exp. 3 (FEATURE_MAP_REPS=1, ANSATZ_REPS=3, 4 qubits, 32 params, readout PauliZ(0), score (1-expval)/2)",
    "change": "only the set of ZZ pairs in the feature map; each excluded pair removes its whole CNOT-RZ-CNOT block",
    "topologies": {t: {"pairs": [list(p) for p in pairs], "zz_interactions": len(pairs),
                       "native_op_order": op_sequence(circuit_ops[t])} for t, pairs in TOPOLOGIES.items()},
    "feature_map_order": "per qubit i: H, RZ(2 x_i); then for (i, j) in pairs, in the official order: CNOT(i,j), RZ(2 (pi - x_i)(pi - x_j)) on j, CNOT(i,j)",
    "ansatz": "unchanged: (ANSATZ_REPS+1) layers of RY, RZ on every qubit, CNOT(i, i+1) between layers",
    "full_weights": "results/result_exp3/weights (not retrained)", "trained_from_scratch": NEW_TOPOLOGIES,
    "seeds": SEEDS, "noise_levels": P_ALL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "optimizer": "Adam", "loss": "BCELoss", "init": "init_weights(seed), identical to Exp. 3",
    "shuffle": "torch.Generator().manual_seed(seed), identical to Exp. 3",
    "balancing": "undersample(X_train, y_train, seed), ratio 1:1", "encoder": "AngleEncoder(4, seed) fitted on the 1:1 set, shared by all topologies",
    "epoch_selection": "best validation AUPRC at p = 0",
    "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed",
    "threshold": "max F1 on validation scores at the same p (selected and evaluated on the same validation set: optimistic)",
    "evaluation_set": "validation (test not used)", "split": "split_indices.npz (verified against official reconstruction)",
    "torch_default_dtype": str(TORCH_DEFAULT_DTYPE), "full_noisy_scores": "recomputed in float64; historical Exp. 3 files not modified",
    "exact_tolerance": EXACT_TOL, "reproduction_tolerance_vs_exp3_float32": REPRO_TOL,
    "shots_per_execution": SHOTS_PER_EXECUTION, "smoke_rows": SMOKE_ROWS, "smoke_epochs": SMOKE_EPOCHS,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
for t in NEW_TOPOLOGIES:
    (WEIGHTS_DIR / t).mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RESULTS_DIR / "config.json"
TRAIN_RUNS_PATH, VAL_RUNS_PATH = RESULTS_DIR / "train_runs.jsonl", RESULTS_DIR / "val_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: mută directorul sau schimbă RESULTS_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")
circuit_comparison.to_csv(RESULTS_DIR / "circuit_comparison.csv")
print("salvat: config.json, circuit_comparison.csv")

## 6. Datele per seed, modelele FULL (Exp. 3) și verificarea buclei de antrenare

Pentru fiecare seed: `undersample` 1:1 și `AngleEncoder(4, seed)`, calculate **o singură dată** și folosite de toate cele trei topologii. FULL folosește ponderile salvate de Exp. 3 (AUPRC fără zgomot reprodus la 1e-12).

**Verificarea buclei:** reantrenăm FULL pentru seed 0 (30 de epoci, rezultat folosit doar pentru verificare, nesalvat ca model) și cerem ca istoricul (loss, AUPRC pe epoci), epoca aleasă și ponderile să coincidă cu cele salvate de Exp. 3. Dacă trece, STAR / LINEAR sunt antrenate exact în condițiile Exp. 3: aceeași inițializare, aceeași ordine a mini-batch-urilor, aceleași date.

In [ ]:
seed_data = {}
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    assert len(y_s) == 590 and int(y_s.sum()) == 295 and X_s.index.is_unique and set(X_s.index) <= set(X_train.index)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)
    seed_data[seed] = {"Z_tr": encoder.transform(X_s), "y_tr": y_s, "Z_val": encoder.transform(X_val)}
    seed_data[seed]["Z_sha256"] = hashlib.sha256(seed_data[seed]["Z_tr"].tobytes() + seed_data[seed]["Z_val"].tobytes()).hexdigest()

models = {t: {} for t in TOPOLOGIES}
for seed in SEEDS:
    saved = torch.load(EXP3_DIR / "weights" / f"seed_{seed}.pt", weights_only=True)
    weights = saved["weights"].detach().clone()
    assert weights.shape == (ANSATZ_REPS + 1, N_QUBITS, 2) and weights.dtype == torch.float64
    assert torch.equal(saved["initial_weights"], init_weights(seed).detach()), f"seed {seed}: inițializarea Exp. 3 ≠ init_weights(seed)"
    t0 = time.perf_counter()
    clean = predict(seed_data[seed]["Z_val"], weights, MODELS["full"])
    elapsed = time.perf_counter() - t0
    ap = average_precision_score(y_val, clean)
    assert abs(ap - saved["best_val_ap"]) < EXACT_TOL, f"seed {seed}: {ap} vs {saved['best_val_ap']}"
    models["full"][seed] = {"weights": weights, "best_epoch": saved["best_epoch"], "best_val_ap": saved["best_val_ap"],
                            "history": saved["history"], "clean": clean, "clean_time": elapsed}
    print(f"FULL seed {seed}: ponderi Exp. 3 încărcate   epoca {saved['best_epoch']:2d}   val AUPRC (p = 0) {ap:.6f} = salvat {saved['best_val_ap']:.6f}")

# verificarea buclei de antrenare: FULL, seed 0, reantrenat și comparat cu Exp. 3
s0 = SEEDS[0]
check_run = train_vqc_run(seed_data[s0]["Z_tr"], seed_data[s0]["y_tr"], seed_data[s0]["Z_val"], y_val, init_weights(s0), s0, MODELS["full"])
hist_err = max(max(abs(a["train loss"] - b["train loss"]), abs(a["validation AUPRC"] - b["validation AUPRC"]))
               for a, b in zip(check_run["history"], models["full"][s0]["history"]))
w_err = (check_run["weights"] - models["full"][s0]["weights"]).abs().max().item()
assert check_run["best_epoch"] == models["full"][s0]["best_epoch"] and hist_err < EXACT_TOL and w_err < EXACT_TOL, (hist_err, w_err)
FULL_BATCH_ORDER_SHA = check_run["batch_order_sha256"]
FULL_RETRAIN_CHECK_TIME = check_run["time_train_s"]
checks.append({"verificare": "bucla de antrenare: FULL seed 0 reantrenat = Exp. 3 (istoric 30 epoci, epoca aleasă, ponderi)",
               "intrări": EPOCHS, "trecut": True, "eroare max": max(hist_err, w_err)})
print(f"✅ bucla de antrenare reproduce Exp. 3 (seed 0): eroare max istoric {hist_err:.1e}, ponderi {w_err:.1e}, "
      f"epoca {check_run['best_epoch']} ({FULL_RETRAIN_CHECK_TIME:.0f} s, nesalvat ca model)")

## 7. Test de fum: STAR și LINEAR (seed 0)

2 epoci de antrenare (nesalvate) și o trecere cu zgomot p = 0.05 pe primele 2 000 de tranzacții din validation, ca să verificăm că totul rulează și să estimăm timpul.

In [ ]:
smoke_rows = []
for t in NEW_TOPOLOGIES:
    d = seed_data[s0]
    run = train_vqc_run(d["Z_tr"], d["y_tr"], d["Z_val"], y_val, init_weights(s0), s0, MODELS[t], epochs=SMOKE_EPOCHS)
    t0 = time.perf_counter()
    s_noisy = noisy_predict(d["Z_val"][:SMOKE_ROWS], run["weights"], 0.05, CIRCUITS[t])
    noisy_time = time.perf_counter() - t0
    s_clean = predict(d["Z_val"][:SMOKE_ROWS], run["weights"], MODELS[t])
    s_mixed0 = noisy_predict(d["Z_val"][:SMOKE_ROWS], run["weights"], 0.0, CIRCUITS[t])
    sanity = np.abs(s_mixed0 - s_clean).max()
    assert sanity < 1e-10, f"{t}: default.mixed la p = 0 ≠ default.qubit ({sanity})"
    assert np.isfinite(s_noisy).all() and np.isfinite(run["history"][-1]["train loss"])
    smoke_rows.append({"topologie": t, f"timp antrenare {SMOKE_EPOCHS} epoci [s]": run["time_train_s"],
                       "loss epoca 2": run["history"][-1]["train loss"], "val AUPRC epoca 2": run["history"][-1]["validation AUPRC"],
                       f"timp p = 0.05, {SMOKE_ROWS} tranz. [s]": noisy_time, "default.mixed p = 0 vs default.qubit": sanity,
                       "MAE p = 0.05 (smoke)": np.abs(s_noisy - s_clean).mean()})
    checks.append({"verificare": f"test de fum {t}: default.mixed p = 0 = default.qubit", "intrări": SMOKE_ROWS, "trecut": True, "eroare max": sanity})
smoke = pd.DataFrame(smoke_rows).set_index("topologie")
display(smoke)
est_train = smoke[f"timp antrenare {SMOKE_EPOCHS} epoci [s]"].sum() / SMOKE_EPOCHS * EPOCHS * len(SEEDS) / 60
est_noise = smoke[f"timp p = 0.05, {SMOKE_ROWS} tranz. [s]"].mean() * len(y_val) / SMOKE_ROWS * len(NOISE_LEVELS) * len(SEEDS) * len(TOPOLOGIES) / 60
print(f"estimare: antrenare STAR + LINEAR ~{est_train:.0f} min, evaluare cu zgomot (3 topologii) ~{est_noise:.0f} min")

## 8. Antrenarea STAR și LINEAR (5 seeds, 30 de epoci, de la zero)

Aceeași buclă ca la Exp. 3, aceeași inițializare `init_weights(seed)` și aceeași ordine a mini-batch-urilor (hash SHA-256 al celor 30 de permutări, comparat cu cel al FULL pentru seed 0). Ponderile epocii alese se salvează în `weights/<topologie>/seed_<s>.pt` imediat după fiecare antrenare, iar rezumatul în `train_runs.jsonl`; o rulare întreruptă reia de unde a rămas.

In [ ]:
train_done = {(r["topologie"], r["seed"]): r for r in load_jsonl(TRAIN_RUNS_PATH)}
for t in NEW_TOPOLOGIES:
    for seed in SEEDS:
        d = seed_data[seed]
        path = WEIGHTS_DIR / t / f"seed_{seed}.pt"
        if path.exists() and (t, seed) in train_done:
            saved = torch.load(path, weights_only=True)
            source = "încărcat"
        else:
            initial = init_weights(seed)
            run = train_vqc_run(d["Z_tr"], d["y_tr"], d["Z_val"], y_val, initial, seed, MODELS[t])
            saved = {"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"],
                     "history": run["history"], "initial_weights": initial.detach().clone()}
            torch.save(saved, path)
            losses = [h["train loss"] for h in run["history"]]
            append_jsonl(TRAIN_RUNS_PATH, {
                "topologie": t, "seed": seed, "best_epoch": run["best_epoch"], "val AUPRC (p = 0)": run["best_val_ap"],
                "train loss la epoca aleasă": losses[run["best_epoch"] - 1], "train loss final": losses[-1], "train loss minim": min(losses),
                "epoci executate": len(run["history"]), "exemple train": len(d["y_tr"]), "timp antrenare [s]": run["time_train_s"],
                "sha256 ordine mini-batch": run["batch_order_sha256"], "sha256 unghiuri (train + val)": d["Z_sha256"]})
            train_done[(t, seed)] = load_jsonl(TRAIN_RUNS_PATH, required=True)[-1]
            if seed == s0:
                assert run["batch_order_sha256"] == FULL_BATCH_ORDER_SHA, "ordinea mini-batch-urilor diferă de FULL / Exp. 3"
            source = f"antrenat și salvat ({run['time_train_s']:.0f} s)"
        assert torch.equal(saved["initial_weights"], init_weights(seed).detach()), f"{t} seed {seed}: altă inițializare decât Exp. 3"
        assert train_done[(t, seed)]["sha256 unghiuri (train + val)"] == d["Z_sha256"], f"{t} seed {seed}: alte unghiuri"
        assert len(saved["history"]) == EPOCHS and saved["best_epoch"] == 1 + int(np.argmax([h["validation AUPRC"] for h in saved["history"]]))
        t0 = time.perf_counter()
        clean = predict(d["Z_val"], saved["weights"], MODELS[t])
        elapsed = time.perf_counter() - t0
        assert abs(average_precision_score(y_val, clean) - saved["best_val_ap"]) < EXACT_TOL, f"{t} seed {seed}: ponderile nu reproduc AUPRC-ul salvat"
        models[t][seed] = {"weights": saved["weights"], "best_epoch": saved["best_epoch"], "best_val_ap": saved["best_val_ap"],
                           "history": saved["history"], "clean": clean, "clean_time": elapsed}
        print(f"{t:<6} seed {seed}: {source}   epoca aleasă {saved['best_epoch']:2d}   val AUPRC (p = 0) {saved['best_val_ap']:.4f}"
              f"   (FULL {models['full'][seed]['best_val_ap']:.4f})")
orders = {r["sha256 ordine mini-batch"] for (t, seed), r in train_done.items() if seed == s0}
assert orders == {FULL_BATCH_ORDER_SHA}
print("✅ STAR și LINEAR: 5 seeds × 30 epoci, ponderi salvate; aceeași inițializare, aceleași unghiuri și aceeași ordine a mini-batch-urilor ca Exp. 3")

## 9. Evaluarea sub zgomot pe validation (toate cele trei topologii)

Pentru fiecare seed, topologie și p: scorurile pe toată validarea (fără zgomot cu `predict`, cu zgomot cu `noisy_predict` oficial), pragul max-F1 pe validation **la același p**, metricile la acest prag, Spearman ρ, MAE / RMSE / max |Δ| între scorurile cu și fără zgomot ale **aceluiași** model, timpul de simulare. Fiecare (topologie, seed, p) terminat se salvează imediat în `val_runs.jsonl`.

FULL se recalculează în float64 și se verifică față de Exp. 8 (același calcul, float64, la 1e-12) și față de Exp. 3 istoric (float32, la 1e-6).

In [ ]:
exp3_val = {(r["seed"], r["p"]): r for r in load_jsonl(EXP3_DIR / "val_runs.jsonl", required=True)}
exp8_val = {(r["seed"], r["p"]): r for r in load_jsonl(EXP8_DIR / "val_runs.jsonl", required=True) if r["scenariu"] == "A" and r["circuit"] == "exp3"}


def run_record(t, seed, p, scores, elapsed):
    s0_scores = models[t][seed]["clean"]
    threshold = choose_threshold(y_val, scores)
    diff = scores - s0_scores
    record = {"topologie": t, "seed": seed, "p": p, **evaluate(y_val, scores, threshold), "threshold": float(threshold),
              "Spearman rho": spearmanr(s0_scores, scores).statistic, "MAE scor": np.abs(diff).mean(),
              "RMSE scor": float(np.sqrt(np.mean(diff ** 2))), "max |Δ scor|": np.abs(diff).max(),
              "|⟨Z0⟩| mediu": float(np.mean(np.abs(1 - 2 * scores))), "timp scorare [s]": elapsed,
              "CNOT": int(STRUCT.loc[t, "CNOT"]), "locuri de zgomot": int(STRUCT.loc[t, "locuri de zgomot"])}
    if t == "full":
        ref3, ref8 = exp3_val[(seed, p)], exp8_val[(seed, p)]
        record.update({"|Δ AUPRC| vs Exp. 3 salvat (float32)": abs(record["AUPRC"] - ref3["AUPRC"]),
                       "|Δ prag| vs Exp. 3 salvat (float32)": abs(threshold - ref3["threshold"]),
                       "|Δ AUPRC| vs Exp. 8 (float64)": abs(record["AUPRC"] - ref8["AUPRC"]),
                       "|Δ prag| vs Exp. 8 (float64)": abs(threshold - ref8["prag reoptimizat"]),
                       "|Δ MAE| vs Exp. 8 (float64)": abs(record["MAE scor"] - ref8["MAE scor"]),
                       "|Δ F1| vs Exp. 8 (float64)": abs(record["F1"] - ref8["F1 [prag reoptimizat]"])})
    return record


done = {(r["topologie"], r["seed"], r["p"]) for r in load_jsonl(VAL_RUNS_PATH)}
todo = [(seed, t, p) for seed in SEEDS for t in TOPOLOGIES for p in P_ALL if (t, seed, p) not in done]
print(f"(topologie, seed, p) de evaluat: {len(todo)} din {len(SEEDS) * len(TOPOLOGIES) * len(P_ALL)}")
for seed, t, p in todo:
    m = models[t][seed]
    if p == 0.0:
        scores, elapsed = m["clean"], m["clean_time"]
    else:
        t0 = time.perf_counter()
        scores = noisy_predict(seed_data[seed]["Z_val"], m["weights"], p, CIRCUITS[t])
        elapsed = time.perf_counter() - t0
    record = run_record(t, seed, p, scores, elapsed)
    append_jsonl(VAL_RUNS_PATH, record)
    print(f"{t:<6} seed {seed}  p = {p:<5}  AUPRC {record['AUPRC']:.4f}  F1 {record['F1']:.3f}  TP {record['TP']:2d}  FP {record['FP']:3d}  "
          f"rho {record['Spearman rho']:.4f}  MAE {record['MAE scor']:.4f}  ({elapsed:.0f} s)")

### Integritatea rezultatelor și reproducerea FULL

In [ ]:
runs = pd.DataFrame(load_jsonl(VAL_RUNS_PATH, required=True))
keys = list(zip(runs["topologie"], runs["seed"], runs["p"]))
assert len(keys) == len(set(keys)), "rezultate duplicate în val_runs.jsonl"
missing = {(t, s, p) for t in TOPOLOGIES for s in SEEDS for p in P_ALL} - set(keys)
assert not missing, f"lipsesc: {sorted(missing)}"
runs = runs.sort_values(["topologie", "p", "seed"]).reset_index(drop=True)

repro_cols = [c for c in runs.columns if c.startswith("|Δ") and "vs Exp." in c]
repro = runs[runs["topologie"] == "full"][["seed", "p", *repro_cols]].astype(float)
display(repro.groupby("p")[repro_cols].max())
exact_cols = [c for c in repro_cols if "Exp. 8" in c]
assert repro[exact_cols].max().max() < EXACT_TOL, "FULL nu reproduce Exp. 8 (float64)"
assert repro[[c for c in repro_cols if "Exp. 3" in c]].max().max() < REPRO_TOL, "FULL nu reproduce Exp. 3 istoric"
checks.append({"verificare": "FULL sub zgomot = Exp. 8 Scenariul A (float64): AUPRC, prag, MAE, F1", "intrări": len(repro), "trecut": True,
               "eroare max": float(repro[exact_cols].max().max())})
checks.append({"verificare": "FULL sub zgomot = Exp. 3 istoric (float32): AUPRC, prag", "intrări": len(repro), "trecut": True,
               "eroare max": float(repro[[c for c in repro_cols if 'Exp. 3' in c]].max().max())})
pd.DataFrame(checks).to_csv(RESULTS_DIR / "checks.csv", index=False)
print(f"✅ FULL recalculat reproduce Exp. 8 (float64) la {EXACT_TOL:.0e} și Exp. 3 istoric (float32) la {REPRO_TOL:.0e}; {len(checks)} verificări salvate în checks.csv")

## 10. Rezultate: medie ± std pe 5 seeds, impactul zgomotului, diferențe pereche față de FULL

Diferențele sunt **pereche**: același seed, aceleași unghiuri, aceeași inițializare și aceeași ordine a mini-batch-urilor; diferă doar topologia ZZ (și, în consecință, ponderile antrenate). Precision / recall / F1 / MCC sunt optimiste (prag ales și evaluat pe același validation); AUPRC, AUROC și ρ nu depind de prag.

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN"]
PERTURBATION = ["MAE scor", "RMSE scor", "max |Δ scor|", "|⟨Z0⟩| mediu"]
LOWER_IS_BETTER = {"MAE scor", "RMSE scor", "max |Δ scor|", "FP", "FN", "timp scorare [s]", "pierdere AUPRC față de p = 0"}
COUNTS = {"TP", "FP", "FN", "TN"}


def mean_std(values, digits=4):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def per_seed(t, p, metric):
    sel = runs[(runs["topologie"] == t) & (runs["p"] == p)].set_index("seed").loc[SEEDS]
    if metric == "pierdere AUPRC față de p = 0":
        return per_seed(t, 0.0, "AUPRC") - sel["AUPRC"].astype(float)
    return sel[metric].astype(float)


val_summary = pd.DataFrame([
    {"topologie": TOPO_LABELS[t], "p": p, **{m: mean_std(per_seed(t, p, m), 1 if m in COUNTS else 4) for m in METRICS},
     "prag": mean_std(per_seed(t, p, "threshold"))}
    for t in TOPOLOGIES for p in P_ALL
]).set_index(["topologie", "p"])
val_summary.to_csv(RESULTS_DIR / "val_summary.csv")
print("Validation, medie ± std pe 5 seeds (prag max-F1 la același p):")
display(val_summary[["AUPRC", "AUROC", "Spearman rho", "F1", "MCC", "TP", "FP"]])

impact_rows = []
for t in TOPOLOGIES:
    for p in P_ALL:
        auprc = per_seed(t, p, "AUPRC")
        loss = per_seed(t, p, "pierdere AUPRC față de p = 0")
        impact_rows.append({"topologie": TOPO_LABELS[t], "p": p, "CNOT": int(STRUCT.loc[t, "CNOT"]),
                            "locuri de zgomot": int(STRUCT.loc[t, "locuri de zgomot"]), "adâncime": int(STRUCT.loc[t, "adâncime"]),
                            **{m: mean_std(per_seed(t, p, m), 6) for m in PERTURBATION},
                            "factor de contracție |⟨Z0⟩|(p) / |⟨Z0⟩|(0)": mean_std(per_seed(t, p, "|⟨Z0⟩| mediu") / per_seed(t, 0.0, "|⟨Z0⟩| mediu")),
                            "Spearman rho": mean_std(per_seed(t, p, "Spearman rho")),
                            "pierdere AUPRC față de p = 0 (absolută)": mean_std(loss),
                            "pierdere AUPRC față de p = 0 (relativă)": mean_std(loss / per_seed(t, 0.0, "AUPRC")),
                            "AUPRC: std între seeds": auprc.std(ddof=1), "AUPRC: min seed": auprc.min(), "AUPRC: max seed": auprc.max()})
noise_impact = pd.DataFrame(impact_rows).set_index(["topologie", "p"])
noise_impact.to_csv(RESULTS_DIR / "noise_impact.csv")
print("Impactul zgomotului (scoruri cu zgomot vs fără zgomot ale aceluiași model):")
display(noise_impact)

PAIRED_METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP",
                  "MAE scor", "RMSE scor", "pierdere AUPRC față de p = 0", "timp scorare [s]"]
paired_rows, paired_seed_rows = [], []
for t in NEW_TOPOLOGIES:
    for p in P_ALL:
        for m in PAIRED_METRICS:
            full, new = per_seed("full", p, m), per_seed(t, p, m)
            d = new - full
            better = (d < 0) if m in LOWER_IS_BETTER else (d > 0)
            paired_rows.append({"comparație": f"{t.upper()} − FULL", "p": p, "metrică": m, "FULL": mean_std(full), t.upper(): mean_std(new),
                                "Δ medie": d.mean(), "Δ std": d.std(ddof=1), "Δ (medie ± std)": mean_std(d),
                                "seeds cu topologia nouă mai bună": int(better.sum()), "seeds egale": int((d == 0).sum())})
            for seed in SEEDS:
                paired_seed_rows.append({"comparație": f"{t.upper()} − FULL", "p": p, "metrică": m, "seed": seed,
                                         "FULL": full[seed], "nou": new[seed], "Δ": d[seed]})
paired = pd.DataFrame(paired_rows)
paired_seeds = pd.DataFrame(paired_seed_rows)
paired_seeds.to_csv(RESULTS_DIR / "paired_differences.csv", index=False)
print("Diferențe pereche AUPRC per seed:")
display(paired_seeds[paired_seeds["metrică"] == "AUPRC"].pivot_table(index=["comparație", "p"], columns="seed", values="Δ").round(4))

## 11. Comparația cu Exp. 3 și cost ledger

**Costul are patru mărimi diferite, raportate separat:**
1. **Structura per execuție** (porți, CNOT, adâncime, locuri de zgomot): cât zgomot acumulează și cât durează un shot pe hardware.
2. **Execuții de circuit** (nominal, formula Părții 8: antrenare = epoci × 590 × (1 + 2P); validare după fiecare epocă; trecerea finală pe validation la p = 0 o dată + câte una per p cu zgomot) și varianta măsurată cu `qml.Tracker`. Ele depind de parametri, exemple, epoci și p, **nu de porți**.
3. **Shots nominale** = execuții × 1024. **Nu** au fost trase: simularea este exactă (`shots=None`).
4. **Timpul simulatorului**, informativ (depinde de mașină, nu este cost hardware).

Aplicările CNOT = execuții × CNOT per execuție: o mărime derivată care combină (1) și (2), utilă pentru comparația pe hardware unde porțile de doi qubiți sunt cele mai scumpe și mai zgomotoase.

In [ ]:
comparison_rows = []
for t in NEW_TOPOLOGIES:
    for key in ["interacțiuni ZZ", "porți", "porți 1 qubit", "CNOT", "adâncime", "locuri de zgomot", "parametri antrenabili",
                "execuții / exemplu (Tracker, parameter shift)", "parametri cu gradient nenul"]:
        f, n = STRUCT.loc["full", key], STRUCT.loc[t, key]
        comparison_rows.append({"comparație": f"{t.upper()} − FULL", "metrică": key, "FULL (Exp. 3)": f"{f:g}", t.upper(): f"{n:g}",
                                "diferență": f"{n - f:+g} ({(n - f) / f:+.1%})" if f else f"{n - f:+g}", "seeds cu topologia nouă mai bună": "structural"})
    for m, p in [("AUPRC", 0.0), ("AUPRC", 0.001), ("AUPRC", 0.01), ("AUPRC", 0.05), ("AUROC", 0.05), ("Spearman rho", 0.01), ("Spearman rho", 0.05),
                 ("MAE scor", 0.01), ("MAE scor", 0.05), ("RMSE scor", 0.05), ("pierdere AUPRC față de p = 0", 0.05),
                 ("F1", 0.0), ("F1", 0.05), ("MCC", 0.05), ("TP", 0.05), ("FP", 0.05), ("timp scorare [s]", 0.05)]:
        row = paired[(paired["comparație"] == f"{t.upper()} − FULL") & (paired["p"] == p) & (paired["metrică"] == m)].iloc[0]
        comparison_rows.append({"comparație": row["comparație"], "metrică": f"{m} la p = {p}", "FULL (Exp. 3)": row["FULL"], t.upper(): row[t.upper()],
                                "diferență": row["Δ (medie ± std)"], "seeds cu topologia nouă mai bună": f"{row['seeds cu topologia nouă mai bună']} / {len(SEEDS)}"})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RESULTS_DIR / "comparison_vs_exp3.csv", index=False)
for t in NEW_TOPOLOGIES:
    print(f"{TOPO_LABELS[t]} vs FULL (Exp. 3) — validation, medie ± std pe 5 seeds, diferențe pereche")
    display(comparison[comparison["comparație"] == f"{t.upper()} − FULL"].drop(columns="comparație").set_index("metrică").dropna(axis=1, how="all"))

n_val = len(y_val)
train_records = {(r["topologie"], r["seed"]): r for r in load_jsonl(TRAIN_RUNS_PATH, required=True)}
exp3_train = {r["seed"]: r for r in load_jsonl(EXP3_DIR / "train_runs.jsonl", required=True)}
ledger_rows = []
for t in TOPOLOGIES:
    epochs = {len(models[t][s]["history"]) for s in SEEDS}
    assert epochs == {EPOCHS}
    tracker = STRUCT.loc[t, "execuții / exemplu (Tracker, parameter shift)"]
    cnot = int(STRUCT.loc[t, "CNOT"])
    components = [
        ("1. antrenare: forward + parameter shift (nominal 1 + 2P)", EPOCHS * 590 * EXECUTIONS_PER_SAMPLE),
        ("1′. antrenare: forward + parameter shift (Tracker)", EPOCHS * 590 * tracker),
        ("2. validare după fiecare epocă, p = 0", EPOCHS * n_val),
        ("3a. validation p = 0, ponderile alese (o dată)", n_val),
        ("3b. validation sub zgomot, p = 0.001 / 0.01 / 0.05", len(NOISE_LEVELS) * n_val),
    ]
    for name, per_seed_exec in components:
        ledger_rows.append({"topologie": TOPO_LABELS[t], "componentă": name,
                            "stare": ("refolosit din Exp. 3 (nereantrenat în Exp. 9)" if t == "full" and name[0] in "12" else "executat pe simulator în Exp. 9"),
                            "execuții / seed": per_seed_exec, f"execuții {len(SEEDS)} seeds": per_seed_exec * len(SEEDS),
                            f"shots nominale {len(SEEDS)} seeds": per_seed_exec * len(SEEDS) * SHOTS_PER_EXECUTION,
                            "CNOT / execuție": cnot, "aplicări CNOT / seed (derivat)": per_seed_exec * cnot,
                            "porți / execuție": int(STRUCT.loc[t, "porți"]), "adâncime / execuție": int(STRUCT.loc[t, "adâncime"])})
    nominal_total = sum(v for n, v in components if not n.startswith("1′"))
    tracker_total = sum(v for n, v in components if not n.startswith("1."))
    for name, total in [("TOTAL antrenare + validare (nominal)", nominal_total), ("TOTAL antrenare + validare (Tracker)", tracker_total)]:
        ledger_rows.append({"topologie": TOPO_LABELS[t], "componentă": name, "stare": "sumă", "execuții / seed": total,
                            f"execuții {len(SEEDS)} seeds": total * len(SEEDS), f"shots nominale {len(SEEDS)} seeds": total * len(SEEDS) * SHOTS_PER_EXECUTION,
                            "CNOT / execuție": cnot, "aplicări CNOT / seed (derivat)": total * cnot,
                            "porți / execuție": int(STRUCT.loc[t, "porți"]), "adâncime / execuție": int(STRUCT.loc[t, "adâncime"])})
cost_ledger = pd.DataFrame(ledger_rows).set_index(["topologie", "componentă"])
exp3_ledger = pd.read_csv(EXP3_DIR / "cost_ledger.csv", index_col=0)
assert cost_ledger.loc[(TOPO_LABELS["full"], "1. antrenare: forward + parameter shift (nominal 1 + 2P)"), "execuții / seed"] == exp3_ledger.iloc[0]["execuții / seed, Exp. 3 (ZZ ×1)"]
cost_ledger.to_csv(RESULTS_DIR / "cost_ledger.csv")
print(f"Cost ledger (nominal: parameter shift, {SHOTS_PER_EXECUTION} shots / execuție; shots NU au fost trase în simulare):")
display(cost_ledger.map(lambda v: f"{v:,.0f}" if isinstance(v, (int, float, np.integer, np.floating)) else v))

sim_rows = []
for t in TOPOLOGIES:
    if t == "full":
        train_times = [exp3_train[s]["timp antrenare [s]"] for s in SEEDS]
        train_source = "Exp. 3 (train_runs.jsonl, altă sesiune)"
    else:
        train_times = [train_records[(t, s)]["timp antrenare [s]"] for s in SEEDS]
        train_source = "Exp. 9"
    noisy = runs[(runs["topologie"] == t) & (runs["p"] > 0)].groupby("seed")["timp scorare [s]"].sum().loc[SEEDS]
    sim_rows.append({"topologie": TOPO_LABELS[t], "antrenare: sursă": train_source, "antrenare 5 seeds [min]": sum(train_times) / 60,
                     "antrenare / seed [s]": mean_std(train_times, 1),
                     "scorare fără zgomot validation / seed [s]": mean_std([models[t][s]["clean_time"] for s in SEEDS], 2),
                     "scorare cu zgomot (3 p) 5 seeds [min]": noisy.sum() / 60, "scorare cu zgomot (3 p) / seed [s]": mean_std(noisy, 1)})
sim_rows.append({"topologie": TOPO_LABELS["full"], "antrenare: sursă": "Exp. 9: verificarea buclei, seed 0 (nesalvat)",
                 "antrenare 5 seeds [min]": np.nan, "antrenare / seed [s]": f"{FULL_RETRAIN_CHECK_TIME:.1f}"})
simulator_time = pd.DataFrame(sim_rows).set_index(["topologie", "antrenare: sursă"])
simulator_time.to_csv(RESULTS_DIR / "simulator_time.csv")
print("Timp efectiv de simulare (informativ; NU este cost hardware):")
display(simulator_time)

## 12. Grafice

In [ ]:
draw_x, draw_w = torch.tensor(seed_data[s0]["Z_val"][0]), models["full"][s0]["weights"]
circuit_text = []
for k, (t, fn) in enumerate(CIRCUITS.items(), start=1):
    fig, ax = qml.draw_mpl(qml.QNode(fn, dev4), decimals=2, style="pennylane")(draw_x, models[t][s0]["weights"])
    fig.suptitle(f"{TOPO_LABELS[t]} · perechi ZZ {STRUCT.loc[t, 'perechi ZZ']} · {int(STRUCT.loc[t, 'porți'])} porți, "
                 f"{int(STRUCT.loc[t, 'CNOT'])} CNOT, adâncime {int(STRUCT.loc[t, 'adâncime'])}, {int(STRUCT.loc[t, 'locuri de zgomot'])} locuri de zgomot "
                 f"(seed 0, prima tranzacție din validation)", fontsize=14)
    fig.savefig(PLOTS_DIR / f"1{'abc'[k - 1]}_circuit_{t}.png", dpi=110, bbox_inches="tight")
    plt.show()
    circuit_text.append(f"=== {TOPO_LABELS[t]} ===\n" + qml.draw(qml.QNode(fn, dev4), decimals=2, max_length=250)(draw_x, models[t][s0]["weights"]))
(PLOTS_DIR / "1_circuits.txt").write_text("\n\n".join(circuit_text), encoding="utf-8")
print(circuit_text[1])

In [ ]:
TOPS = list(TOPOLOGIES)
xt = np.arange(len(TOPS))
xlabels_topo = [TOPO_LABELS[t].replace(" (Exp. 3)", "\n(Exp. 3)") for t in TOPS]


def bar_labels(ax, bars, values, fmt="{:.0f}"):
    for bar, v in zip(bars, values):
        ax.annotate(fmt.format(v), (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    textcoords="offset points", xytext=(0, 3), ha="center", fontsize=10, color="0.2")


# 2-4. CNOT, locuri de zgomot, adâncime: un fișier per mărime (feature map + ansatz suprapuse)
for num, key, ylabel, title in [(2, "CNOT", "CNOT", "CNOT per circuit"),
                                (3, "locuri de zgomot", "canale depolarizante", "Locuri de zgomot per circuit"),
                                (4, "adâncime", "adâncime (straturi de porți)", "Adâncimea circuitului")]:
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    total = [int(STRUCT.loc[t, key]) for t in TOPS]
    if key == "adâncime":
        fm = [int(STRUCT.loc[t, "feature map: adâncime"]) for t in TOPS]
        bars = ax.bar(xt, total, width=0.55, color=[TOPO_COLORS[t] for t in TOPS])
        ax.bar(xt, fm, width=0.55, color="white", alpha=0.35, hatch="//", edgecolor="white", label="din care: adâncimea feature map-ului singur")
    else:
        fm = [int(STRUCT.loc[t, f"feature map: {key}"]) for t in TOPS]
        an = [int(STRUCT.loc[t, f"ansatz: {key}"]) for t in TOPS]
        ax.bar(xt, an, width=0.55, color=CNOT_COLOR, label="ansatz (identic în toate)")
        bars = ax.bar(xt, fm, width=0.55, bottom=an, color=[TOPO_COLORS[t] for t in TOPS], edgecolor="white", linewidth=2, label="feature map (ZZ), în culoarea topologiei")
        for x, f, a in zip(xt, fm, an):
            ax.annotate(f"{f}", (x, a + f / 2), ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    for x, v in zip(xt, total):
        ax.annotate(f"{v}", (x, v), textcoords="offset points", xytext=(0, 3), ha="center", fontsize=11, color="0.15", fontweight="bold")
    ax.set_xticks(xt, xlabels_topo)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(total) * 1.15)
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8, framealpha=0.95)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"{num}_{key.replace(' ', '_')}.png", dpi=130)
    plt.show()

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels_p = ["0", *[str(p) for p in NOISE_LEVELS]]
OFFSETS = {"full": -0.1, "star": 0.0, "linear": 0.1}


def plot_vs_p(metric, ylabel, title, filename, legend_loc="best"):
    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    for t in TOPS:
        values = [per_seed(t, p, metric) for p in P_ALL]
        for i, v in enumerate(values):
            ax.scatter(np.full(len(v), xpos[i] + OFFSETS[t]), v, s=12, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
        ax.errorbar(xpos + OFFSETS[t], [v.mean() for v in values], yerr=[v.std(ddof=1) for v in values], color=TOPO_COLORS[t],
                    marker=TOPO_MARKERS[t], markersize=8, linewidth=2, capsize=4, zorder=3,
                    label=f"{TOPO_LABELS[t]} · {int(STRUCT.loc[t, 'CNOT'])} CNOT")
    ax.set_xticks(xpos, xlabels_p)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9, loc=legend_loc)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=130)
    plt.show()


plot_vs_p("AUPRC", "AUPRC (validation)", "AUPRC vs p · medie ± std pe 5 seeds (gri: seeds)", "5_auprc_vs_p.png", "lower left")
plot_vs_p("F1", "F1 la pragul max-F1 (același p)", "F1 vs p · medie ± std pe 5 seeds · optimist (prag pe validation)", "6_f1_vs_p.png", "lower left")
plot_vs_p("Spearman rho", "Spearman ρ (scoruri cu vs fără zgomot)", "Spearman ρ vs p · medie ± std pe 5 seeds", "7_spearman_vs_p.png", "lower left")
plot_vs_p("MAE scor", "MAE |scor cu zgomot − scor fără zgomot|", "Perturbarea scorurilor vs p · medie ± std pe 5 seeds", "8_mae_vs_p.png", "upper left")

In [ ]:
# 9. AUPRC per seed la p = 0.05
fig, ax = plt.subplots(figsize=(8.5, 4.4))
width = 0.26
for k, t in enumerate(TOPS):
    v = per_seed(t, 0.05, "AUPRC")
    ax.bar(np.arange(len(SEEDS)) + (k - 1) * width, v.to_numpy(), width=width - 0.02, color=TOPO_COLORS[t],
           label=f"{TOPO_LABELS[t]} (medie {v.mean():.3f})")
ax.set_xticks(range(len(SEEDS)), [f"seed {s}" for s in SEEDS])
ax.set_ylabel("AUPRC la p = 0.05 (validation)")
low = min(per_seed(t, 0.05, "AUPRC").min() for t in TOPS)
ax.set_ylim(max(0, low - 0.1), None)
ax.set_title("AUPRC per seed la p = 0.05")
ax.legend(fontsize=9, loc="lower right", framealpha=0.95)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "9_auprc_per_seed_p0.05.png", dpi=130)
plt.show()

# 10. diferențe pereche AUPRC vs p
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for ax, t in zip(axes, NEW_TOPOLOGIES):
    for seed in SEEDS:
        d = [per_seed(t, p, "AUPRC")[seed] - per_seed("full", p, "AUPRC")[seed] for p in P_ALL]
        ax.plot(xpos, d, color=SEED_DOT_COLOR, marker="o", markersize=4, linewidth=1, alpha=0.8, zorder=2)
        ax.annotate(f"s{seed}", (xpos[-1], d[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
    mean_d = [(per_seed(t, p, "AUPRC") - per_seed("full", p, "AUPRC")).mean() for p in P_ALL]
    ax.plot(xpos, mean_d, color=TOPO_COLORS[t], marker=TOPO_MARKERS[t], markersize=8, linewidth=2.5, label="medie pe 5 seeds", zorder=3)
    ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
    ax.set_xticks(xpos, xlabels_p)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(f"Δ AUPRC ({TOPO_LABELS[t]} − FULL)")
    ax.legend(loc="lower left", fontsize=9)
axes[0].set_ylabel("Δ AUPRC (> 0: topologia rară e mai bună)")
fig.suptitle("Diferențe pereche pe seed (gri: câte o linie per seed) · validation", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "10_paired_delta_auprc.png", dpi=130)
plt.show()

# 11. trade-off locuri de zgomot vs AUPRC (p = 0 → p = 0.05)
fig, ax = plt.subplots(figsize=(7.5, 4.8))
JITTER = {"full": 0.0, "star": -0.8, "linear": 0.8}   # STAR și LINEAR au același număr de locuri de zgomot
for t in TOPS:
    x = STRUCT.loc[t, "locuri de zgomot"] + JITTER[t]
    a0, a5 = per_seed(t, 0.0, "AUPRC"), per_seed(t, 0.05, "AUPRC")
    ax.scatter(np.full(len(a5), x), a5, s=14, color=SEED_DOT_COLOR, alpha=0.7, zorder=2)
    ax.errorbar(x, a0.mean(), yerr=a0.std(ddof=1), fmt=TOPO_MARKERS[t], markersize=9, markerfacecolor="white",
                markeredgecolor=TOPO_COLORS[t], markeredgewidth=2, ecolor=TOPO_COLORS[t], capsize=4, zorder=3)
    ax.errorbar(x, a5.mean(), yerr=a5.std(ddof=1), fmt=TOPO_MARKERS[t], markersize=9, color=TOPO_COLORS[t], capsize=4, zorder=4,
                label=f"{TOPO_LABELS[t]}: {a0.mean():.3f} → {a5.mean():.3f}")
    ax.annotate("", (x, a5.mean()), (x, a0.mean()), arrowprops={"arrowstyle": "->", "color": TOPO_COLORS[t], "lw": 1.2}, zorder=1)
ax.set_xlabel("locuri de zgomot per circuit (STAR / LINEAR: ambele 73, decalate pentru lizibilitate)")
ax.set_ylabel("AUPRC (validation)")
ax.set_title("Trade-off: locuri de zgomot vs AUPRC · gol: p = 0, plin: p = 0.05")
ax.legend(fontsize=9, loc="best")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "11_tradeoff_noise_locations_auprc.png", dpi=130)
plt.show()

# 12. curbe de antrenare
epochs_x = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, key, ylabel in [(axes[0], "train loss", "BCE loss pe train (medie pe epocă)"), (axes[1], "validation AUPRC", "AUPRC pe validation (p = 0)")]:
    for t in TOPS:
        curves = np.array([[h[key] for h in models[t][s]["history"]] for s in SEEDS])
        mean, std = curves.mean(0), curves.std(0, ddof=1)
        ax.fill_between(epochs_x, mean - std, mean + std, color=TOPO_COLORS[t], alpha=0.15, linewidth=0)
        ax.plot(epochs_x, mean, color=TOPO_COLORS[t], linewidth=2, marker=TOPO_MARKERS[t], markersize=4, markevery=5, label=TOPO_LABELS[t])
    ax.set_xlabel("epocă")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
for t in TOPS:
    best = [models[t][s]["best_epoch"] for s in SEEDS]
    axes[1].scatter(best, [models[t][s]["best_val_ap"] for s in SEEDS], s=30, color=TOPO_COLORS[t], marker=TOPO_MARKERS[t],
                    edgecolor="white", linewidth=1, zorder=4)
axes[0].set_title("Train loss pe epoci · medie ± std pe 5 seeds")
axes[1].set_title("AUPRC validation pe epoci (puncte: epoca aleasă per seed)")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "12_training_curves.png", dpi=130)
plt.show()
print("grafice salvate în", PLOTS_DIR.relative_to(REPO_ROOT))

## 13. Cum citim rezultatele

- **Trei niveluri separate:** (1) reducerea **structurală** a expunerii la zgomot (CNOT, locuri de zgomot, adâncime: §4); (2) reducerea **efectivă** a perturbării scorurilor (MAE, RMSE, ρ între scoruri cu și fără zgomot ale aceluiași model: `noise_impact.csv`); (3) păstrarea **capacității de clasificare** (AUPRC la fiecare p, comparat pereche cu FULL).
- O topologie cu mai puține CNOT, dar cu AUPRC mai mic la toate nivelurile de zgomot, **nu** este superioară. O degradare mică pornind de la un AUPRC mic nu se numește robustețe fără această calificare.
- Diferențele sunt pereche (același seed, aceleași date, aceeași inițializare și ordine a mini-batch-urilor). Validation are 99 de fraude; 5 seeds nu dau semnificație statistică formală.
- Pragurile sunt alese și evaluate pe același validation: precision / recall / F1 / MCC / TP / FP sunt optimiste, pentru toate topologiile la fel.
- Costul: execuțiile depind de parametri, exemple, epoci și p, nu de porți; topologia schimbă porțile per execuție, adâncimea și locurile de zgomot.
- Testul nu a fost folosit. Interpretarea și răspunsurile sunt în `results/result_exp9/report.md`.